In [1]:
import pandas as pd

df = pd.read_csv("/Users/julia/Documents/spotify_millsongdata.csv")

if "text" in df.columns:
    # Sprawdź, ile utworów ma brakujące teksty
    missing_lyrics = df["text"].isna().sum()
    total_songs = len(df)

    print(f"Liczba utworów w datasetcie: {total_songs}")
    print(f"Liczba utworów bez tekstu: {missing_lyrics}")
    print(f"Procent utworów bez tekstu: {missing_lyrics / total_songs * 100:.2f}%")
else:
    print("Brak kolumny 'text' – dataset nie zawiera tekstów piosenek.")

Liczba utworów w datasetcie: 57650
Liczba utworów bez tekstu: 0
Procent utworów bez tekstu: 0.00%


In [15]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download('averaged_perceptron_tagger')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Funkcja do mapowania POS na format WordNet
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()  # Pobranie pierwszej litery tagu POS
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)  # Jeśli brak w słowniku, przyjmujemy rzeczownik

def preprocess_text(text):
    if pd.isna(text):
        return ""
    
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Usunięcie liczb
    text = re.sub(r'\W+', ' ', text)  # Usunięcie znaków specjalnych
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]  # Usunięcie stop words
    words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in words]  # Poprawna lematyzacja
    return ' '.join(words)

df['processed_text'] = df['text'].apply(preprocess_text)


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/julia/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [17]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['processed_text'])

# Sprawdzenie wymiarów macierzy
print(X.shape)  # (liczba_piosenek, liczba_słów)


(57650, 5000)


In [19]:
from sklearn.decomposition import LatentDirichletAllocation

# Ustawienie liczby tematów
num_topics = 8

lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
lda.fit(X)

# Pobranie słów kluczowych dla każdego tematu
def display_topics(model, feature_names, num_words):
    for topic_idx, topic in enumerate(model.components_):
        words = [feature_names[i] for i in topic.argsort()[:-num_words - 1:-1]]
        print(f"Temat {topic_idx+1}: {', '.join(words)}")

# Wyświetlenie tematów
display_topics(lda, vectorizer.get_feature_names_out(), 10)


Temat 1: come, rock, home, roll, good, little, old, new, town, man
Temat 2: oh, love, know, say, time, want, need, way, make, feel
Temat 3: love, like, come, feel, dance, tonight, night, blue, make, light
Temat 4: life, let, world, heart, day, god, dream, love, come, know
Temat 5: na, gon, baby, know, wan, want, let, make, cause, ta
Temat 6: la, away, come, time, like, eye, know, turn, fall, life
Temat 7: like, yeah, ya, nigga, fuck, know, em, shit, cause, money
Temat 8: like, girl, say, hey, man, boy, little, look, make, big
